# CEG-WM salient-local-LF mask/write validation

This output-free development Notebook runs exactly two operational preflights and eight fixed scientific mask/write observations. It validates the frozen FLOOR RGB8 quality rule and the nominal masked-LF causal witness; it does not fit the masked-LF whitening asset, execute detection, select a candidate, estimate tau/FPR, or promote the method.

The Notebook only mounts Drive, bridges the required Secrets, checks out the exact execution revision, invokes the repository server, and exports the result or bounded failure ZIP, receipt, and SHA256SUMS before raising on failure.


In [ ]:
from google.colab import drive, userdata
from datetime import datetime, timezone
from hashlib import sha256
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

drive.mount('/content/drive')


In [ ]:
REPOSITORY_URL = 'https://github.com/RICHAAARC/CEG-WM.git'
EXECUTION_REVISION = '7be93ef1e6047907069aab922ee4b32745ea2ce3'
RUN_ID = 'ceg_wm_salient_local_lf_mask_write_registered_dual_carrier_key_correction_validation'
EXPECTED_PACKAGE_SHA256 = 'a0b4480e7d9172ebdfdfd04a1053f916eeba56e24ae3c576d81dcd0c85aed6e4'
EXPECTED_PACKAGE_SIZE_BYTES = 4137274
SESSION_ID = datetime.now(timezone.utc).strftime('colab_%Y%m%dt%H%M%S%fz')
CHECKOUT_ROOT = Path(f'/content/ceg_wm_salient_local_lf_mask_write_checkout_{SESSION_ID}')
DRIVE_MOUNT = Path('/content/drive').resolve()
DRIVE_ROOT = DRIVE_MOUNT / 'MyDrive' / 'CEG-WM' / 'salient_local_lf_mask_write_validation'
PERSISTENT_ROOT = DRIVE_ROOT / 'persistent'
CACHE_ROOT = DRIVE_ROOT / 'cache'
EXPORT_ROOT = DRIVE_ROOT / 'exports' / EXECUTION_REVISION / RUN_ID / SESSION_ID
CHECKPOINT_PATH = DRIVE_MOUNT / 'MyDrive' / 'CEG-WM' / 'models' / 'inspyrenet' / 'ckpt_base.pth'
for required_root in (PERSISTENT_ROOT, CACHE_ROOT):
    required_root.mkdir(parents=True, exist_ok=True)
    assert DRIVE_MOUNT in required_root.resolve().parents
assert CHECKPOINT_PATH.is_file() and not CHECKPOINT_PATH.is_symlink()
assert CHECKPOINT_PATH.name == 'ckpt_base.pth' and CHECKPOINT_PATH.stat().st_size == 367520613
probe_path = PERSISTENT_ROOT / f'.write_probe_{SESSION_ID}'
with probe_path.open('x', encoding='utf-8') as probe:
    probe.write('salient-local-LF mask/write persistent root available\n')
probe_path.unlink()
secret_environment = os.environ.copy()
secret_environment['HF_TOKEN'] = userdata.get('HF_TOKEN')
secret_environment['CEG_WM_ROOT_KEY'] = userdata.get('CEG_WM_ROOT_KEY')
secret_environment['CEG_WM_INSPYRENET_CHECKPOINT_PATH'] = str(CHECKPOINT_PATH)
assert secret_environment['HF_TOKEN'] and secret_environment['CEG_WM_ROOT_KEY']


In [ ]:
CHECKOUT_ROOT.mkdir(parents=True, exist_ok=False)
subprocess.run(['git', '-C', str(CHECKOUT_ROOT), 'init'], check=True)
subprocess.run(['git', '-C', str(CHECKOUT_ROOT), 'remote', 'add', 'origin', REPOSITORY_URL], check=True)
subprocess.run(['git', '-C', str(CHECKOUT_ROOT), 'fetch', '--depth', '1', 'origin', EXECUTION_REVISION], check=True)
subprocess.run(['git', '-C', str(CHECKOUT_ROOT), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
observed_revision = subprocess.run(['git', '-C', str(CHECKOUT_ROOT), 'rev-parse', 'HEAD'], check=True, capture_output=True, text=True).stdout.strip()
observed_status = subprocess.run(['git', '-C', str(CHECKOUT_ROOT), 'status', '--porcelain'], check=True, capture_output=True, text=True).stdout
assert observed_revision == EXECUTION_REVISION and observed_status == ''


In [ ]:
command = [sys.executable, '-m', 'scripts.experiment_execution.salient_local_lf_mask_write_validation_server', '--repository-root', str(CHECKOUT_ROOT), '--expected-revision', EXECUTION_REVISION, '--persistent-root', str(PERSISTENT_ROOT), '--cache-root', str(CACHE_ROOT), '--run-id', RUN_ID, '--session-id', SESSION_ID]
process = subprocess.Popen(command, cwd=CHECKOUT_ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=secret_environment)
assert process.stdout is not None
for log_line in process.stdout:
    print(log_line, end='')
server_exit_code = process.wait()
del secret_environment
receipt_source = PERSISTENT_ROOT / RUN_ID / 'server_receipts' / SESSION_ID / 'execution_receipt.json'
assert receipt_source.is_file(), 'server did not persist a bounded execution receipt'
receipt = json.loads(receipt_source.read_text(encoding='utf-8'))
assert receipt['committed_revision'] == EXECUTION_REVISION
assert receipt['run_id'] == RUN_ID and receipt['session_id'] == SESSION_ID
assert receipt['exit_code'] == server_exit_code
package_available = receipt.get('execution_package_available', True)
if package_available:
    assert receipt['execution_package_sha256'] == EXPECTED_PACKAGE_SHA256
    assert Path(receipt['execution_package_path']).stat().st_size == EXPECTED_PACKAGE_SIZE_BYTES
else:
    assert server_exit_code != 0 and receipt['execution_package_sha256'] is None
    assert receipt['execution_package_relative_path'] is None
assert receipt['operational_unit_count'] == 2 and receipt['scientific_unit_count'] == 8
assert receipt['total_unit_count'] == 10 and receipt['maximum_attempts_per_unit'] == 1


In [ ]:
def file_sha256(path):
    digest = sha256()
    with Path(path).open('rb') as source:
        for block in iter(lambda: source.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def copy_create_only(source, destination, expected_sha256):
    if destination.exists():
        raise RuntimeError('Drive export destination already exists')
    shutil.copyfile(source, destination)
    if file_sha256(destination) != expected_sha256:
        raise RuntimeError('Drive export SHA-256 mismatch')
    return destination

if receipt['artifact_kind'] == 'salient_local_lf_mask_write_validation_startup_failure':
    artifact_source = (PERSISTENT_ROOT / receipt['diagnostic_zip_relative_path']).resolve()
else:
    artifact_source = Path(receipt['artifact_path']).resolve()
assert artifact_source.is_file() and not artifact_source.is_symlink()
assert PERSISTENT_ROOT.resolve() in artifact_source.parents
assert receipt['artifact_kind'] in {'salient_local_lf_mask_write_validation_result', 'salient_local_lf_mask_write_validation_failure', 'salient_local_lf_mask_write_validation_startup_failure'}
assert receipt['formal_tau_created'] is False and receipt['fpr_estimated'] is False and receipt['candidate_promoted'] is False
assert file_sha256(artifact_source) == receipt['artifact_sha256']
receipt_sha256 = file_sha256(receipt_source)
EXPORT_ROOT.mkdir(parents=True, exist_ok=False)
artifact_export = copy_create_only(artifact_source, EXPORT_ROOT / artifact_source.name, receipt['artifact_sha256'])
receipt_export = copy_create_only(receipt_source, EXPORT_ROOT / 'execution_receipt.json', receipt_sha256)
checksums_path = EXPORT_ROOT / 'SHA256SUMS'
with checksums_path.open('x', encoding='utf-8') as checksums:
    checksums.write(f"{receipt['artifact_sha256']}  {artifact_export.name}\n")
    checksums.write(f'{receipt_sha256}  {receipt_export.name}\n')
summary = {'artifact_kind': receipt['artifact_kind'], 'artifact_path': str(artifact_export), 'artifact_sha256': receipt['artifact_sha256'], 'receipt_path': str(receipt_export), 'receipt_sha256': receipt_sha256, 'checksums_path': str(checksums_path), 'committed_revision': EXECUTION_REVISION, 'run_id': RUN_ID, 'session_id': SESSION_ID, 'committed_unit_count': receipt.get('committed_unit_count', 0), 'termination_reason': receipt.get('termination_reason'), 'salient_local_lf_mask_write_aggregate': receipt.get('salient_local_lf_mask_write_aggregate')}
print(json.dumps(summary, indent=2, sort_keys=True))
if server_exit_code != 0:
    raise RuntimeError('salient-local-LF mask/write validation ended with a bounded diagnostic; Drive export is preserved')
